In [ ]:
from importlib import reload
import sys
import subprocess
import copy

import scipy
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../python")
import pymodelica
reload(pymodelica)


# Run the simualtion

pymodelica.run_model() is a wrapper to do a two-step simulation. First, a fast simulation to reach equilibrium. The model is linearized around the equilibrium point. the equilibrium point is also used to initialize a second simulation with fine steps to get a non-linear impulse response.

-overrideFile=C:/Users/Tom/Documents/ThermalModel/parameters/params_lmo_v1_mod_absorber.txt

In [ ]:
MODEL_EXE_NAME = "System_LMO_exe"
MODEL_FILE = "../models/System_LMO.mo"
MODEL_BUILD_DIR = "../build"
MODEL_FILE_LIN = f"linearized_model.py"
MODEL_RESULT = f'{MODEL_BUILD_DIR}/{MODEL_EXE_NAME}_res_init.mat'


OVERRIDE_PARAMS = "../parameters/params_lmo_v1_mod_absorber.txt"
EQUILIBRIUM_TIME = 20.0
EQUILIBRIUM_STEP = 2e-3
FINE_STOP_TIME = 0.4
FINE_STEP_SIZE = 3e-6
TOLERANCE = 1e-8
PULSE_ENERGY_EV = 200

CINDEX_TARGET = 10
CINDEX_TARGETGOLD = 2


In [ ]:
run_info = pymodelica.run_model(
    model_exe = MODEL_EXE_NAME,
    build_dir = MODEL_BUILD_DIR,
    override_file = OVERRIDE_PARAMS,
    params={"Edep": PULSE_ENERGY_EV},
    equilibrium_time=EQUILIBRIUM_TIME,
    coarse_step_size=EQUILIBRIUM_STEP,
    fine_stop_time=FINE_STOP_TIME,
    fine_step_size=FINE_STEP_SIZE,
    tolerance=TOLERANCE,
    solver="dassl",
    linearized_model_file=MODEL_FILE_LIN,
    init_result_file=None,
    pulse_result_file=None,
    keep_runs=True,
    extra_coarse_args=None,
    extra_fine_args=None,
)

print(run_info)
config = pymodelica.read_config(run_info["override_file"])

In [ ]:
# Load simulation result
time_steps_init, data_init = pymodelica.load(run_info["init_result"])

time_steps, data = pymodelica.load(run_info["pulse_result"])
print(data.keys())


# Plot all C and G to check if the system is at equilibrium
y = data_init['T10']*1e3
plt.plot(time_steps_init, y, label=f"Target temperature {y[-1]:.3g} mK")
y = data_init['T1']*1e3
plt.plot(time_steps_init, y, label=f"TES temperature {y[-1]:.3g} mK")
y = data_init['c1.R']*1e3
plt.plot(time_steps_init, y, label=f"TES resistance {y[-1]:.3g} mOhm")
y = data_init['c1.i']*1e6
plt.plot(time_steps_init, y, label=f"TES current {y[-1]:.3g} uA")
y = data_init['c1.i']**2*data_init['c1.R']*1e12
plt.plot(time_steps_init, y, label=f"TES power {y[-1]:.3g} pW")
plt.legend()
plt.xlabel("Time [s]")
plt.grid()
plt.yscale("log")


In [ ]:
# Annotate SVG image with simulation result
pymodelica.mod_svg("../models/System_LMO.svg", "System_LMO_tmp.svg", data_init, system_name="System_LMO", width=1200, display=True)

## Pulse, Noise and Complex impedance

In [ ]:
reload(pymodelica)

In [ ]:
model = pymodelica.TESModel(f"{MODEL_BUILD_DIR}/{MODEL_FILE_LIN}", 
                 MODEL_FILE, 
                 MODEL_RESULT,
                 config = {},
                #  frequencies=np.linspace(1,1e6,10000))
                 f_min = 1, f_max = 1e6, f_scale="log", points = 10_000)

frequencies, noise_current = model.get_noise(
    Rsh_temperature = 0.2,
    # Rp_temperature = 0.2,
    noise_electronics = 4.2e-12,
    noise_flicker_corner = 20,
    noise_flicker_gamma = 1,
    index_cinput = CINDEX_TARGET)

In [ ]:
# Impulse response
t_linear, y_linear = model.get_impulse_split(index_cinput1 = 10, index_cinput2 = 2, 
                              fraction_1 = config["Edep_fractiontarget"], 
                              scale_ev = config["Edep"], 
                              T=np.linspace(0,1,100_000))
mask = t_linear<0.1
y_nonlinear = data['c1.i'] - np.mean(data['c1.i'][:10])

collection_linear = model.get_linear_phonon_energy_collection_efficiency()
collection_nonlinear = model.get_nonlinear_phonon_energy_collection_efficiency(time = time_steps, simulation_result = data, deposited_energy=config["Edep"]*1.6e-19)
plt.plot(time_steps-config["Edep_starttime"], y_nonlinear, label=f"Nonlinear, eff = {100*collection_nonlinear['efficiency']:.2f}%")
plt.plot(t_linear, y_linear[:,1], label=f"linear, eff = {100*collection_linear['efficiency']:.2f}%")
plt.legend()
plt.grid()
plt.xlim(-0.01,0.05)
plt.show()




# Noise
model.plot_noise_current()
fy, y_fft = scipy.signal.periodogram(y_nonlinear*1000, fs=1/(time_steps[1]-time_steps[0]))
plt.plot(fy, np.sqrt(y_fft), label="Pulse non-linear")

fs=1/(t_linear[1]-t_linear[0])
fy, y_fft = scipy.signal.periodogram(y_linear[:,1]*1000, fs=1/(t_linear[1]-t_linear[0]))
plt.plot(fy, np.sqrt(y_fft), label="Pulse linear")
plt.ylim(1e-13, 5e-9)
plt.legend(loc=(1.01,0))



# Resolution
resolution_linear = model.get_resolution(index_cinput = CINDEX_TARGET)
resolution_linear_split = model.get_resolution_split(
    index_cinput1 = CINDEX_TARGET, 
    index_cinput2 = CINDEX_TARGETGOLD, 
    fraction_1 = config["Edep_fractiontarget"])

fy, psd = scipy.signal.periodogram(y_nonlinear/(config["Edep"]*1.6e-19), fs=1/(time_steps[1]-time_steps[0]))
dIdP_nonlinear = np.interp(model.frequencies, fy, np.sqrt(psd/2))
resolution_nonlinear = model.get_resolution(dIdP_nonlinear, n_interp=1000_000)
print(resolution_nonlinear)

print("Target referred resolution[eV]:", resolution_linear)
print("Input referred resolution[eV]:", resolution_linear_split)
print("Input referred resolution(nonlinear)[eV]:", resolution_nonlinear)


# Impedance
fig, axs = model.plot_impedance(fmax=1e5)
# axs[0].set_xlim(right=1e5)
# axs[0].set_ylim(0,0.25)


In [ ]:
fmin_list = [1, 10, 30, 60, 100, 150]
fmax_list = [200, 500, 1e3, 10e3, 30e3, 100e3, 300e3]

resolution_grid = np.zeros((len(fmin_list), len(fmax_list)))
for i,fmin in enumerate(fmin_list):
    for j,fmax in enumerate(fmax_list):
        resolution_grid[i][j] = model.get_resolution(dIdP_nonlinear, fmin = fmin, fmax = fmax, n_interp=1000000)
        

# Format frequency labels for clean display (e.g., 10 kHz instead of 10000.0)
x_labels = [f"{int(f/1e3)} kHz" if f >= 1e3 else f"{f} Hz" for f in fmax_list]
y_labels = [f"{f} Hz" for f in fmin_list]

fig, ax = plt.subplots(figsize=(7, 6))

# Display heatmap (origin='lower' aligns low fmin at the bottom)
im = ax.imshow(resolution_grid, cmap="RdYlGn_r", aspect="auto", origin="lower")

# Set tick locations and labels
ax.set_xticks(np.arange(len(fmax_list)))
ax.set_yticks(np.arange(len(fmin_list)))
ax.set_xticklabels(x_labels)
ax.set_yticklabels(y_labels)

# Axis labels and title
ax.set_xlabel("$f_{max}$", fontsize=12)
ax.set_ylabel("$f_{min}$", fontsize=12)
ax.set_title("Energy resolution vs frequency cutoff", fontsize=14, pad=12)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, cmap="RdYlGn_r")
cbar.set_label("Resolution", rotation=270, labelpad=15)

# Annotate each cell with its value, adjusting text color for readability
threshold = (resolution_grid.max() + resolution_grid.min()) / 3.0
for i in range(len(fmin_list)):
    for j in range(len(fmax_list)):
        val = resolution_grid[i, j]
        text_color = "black" if val > threshold else "white"
        ax.text(
            j,
            i,
            f"{val:.1f}",  # Change to f"{val:.2f}" if values are standard decimals
            ha="center",
            va="center",
            color=text_color,
            fontsize=10,
            fontweight="medium",
        )

plt.tight_layout()
plt.show()


In [ ]:
plt.plot(time_steps, data['T1'])
plt.plot(time_steps, data['T2'])
plt.plot(time_steps, data['T10'])